<a href="https://colab.research.google.com/github/Diagnost13/simulative-git/blob/master/%D0%9F%D0%B5%D1%80%D0%B2%D1%8B%D0%B9_%D0%BA%D0%B5%D0%B9%D1%81.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1 кейс

**Ваша задача написать функцию `process_files`, которая принимает на вход два пути к папкам. Из первой папки необходимо выбрать все "чеки" (файлы по шаблону из условия), а во вторую папку сохранить один объединенный чек (отсортированный по дате, а затем по продукту) под названием `combined_data.csv`.**

**Важно**

Перед началом решения выполните следующую ячейку, чтобы загрузить папку с файлами. После выполнения, в папке `reports_main` будут храниться все присланные магазинами чеки.

In [61]:
!wget https://github.com/vs8th/reports/archive/main.zip

import zipfile

with zipfile.ZipFile("main.zip", 'r') as zip_ref:
    zip_ref.extractall("/content")

!rm main.zip

--2026-03-05 13:09:02--  https://github.com/vs8th/reports/archive/main.zip
Resolving github.com (github.com)... 20.27.177.113
Connecting to github.com (github.com)|20.27.177.113|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://codeload.github.com/Vs8th/reports/zip/refs/heads/main [following]
--2026-03-05 13:09:03--  https://codeload.github.com/Vs8th/reports/zip/refs/heads/main
Resolving codeload.github.com (codeload.github.com)... 20.27.177.114
Connecting to codeload.github.com (codeload.github.com)|20.27.177.114|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3296 (3.2K) [application/zip]
Saving to: ‘main.zip’

main.zip            100%[===================>]   3.22K  --.-KB/s    in 0s      

2026-03-05 13:09:03 (27.7 MB/s) - ‘main.zip’ saved [3296/3296]



Чтобы посмотреть как выглядят подходящие для объединения чеки выполните следующую ячейку.

In [62]:
import pandas as pd

df = pd.read_csv('reports-main/2023-02-17-05-38-2.csv', sep=";")
df

,date,product,store,cost
0,2023-02-17,product_0,store_2,10
1,2023-02-17,product_1,store_2,20
2,2023-02-17,product_2,store_2,30
3,2023-02-17,product_3,store_2,40
4,2023-02-17,product_4,store_2,50


In [63]:
from google.colab import sheets
sheet = sheets.InteractiveSheet(df=df)

https://docs.google.com/spreadsheets/d/1rabyhDx0ztlklYpWV8CLfNeVILqK7_6J03GyW_sGak8/edit#gid=0


**Решение**

Напишите свое решение ниже

**Примечание**

Не все файлы подходящие по названию, будут подходить по содержанию. Там может оказаться лишний столбец, например. Ориентируйтесь на столбцы из чека выше - это то, что вас интересует. Остальные столбцы можно просто отбросить.

**Важно**: разделителем файла на выходе должна быть запятая.

In [64]:
import os
import pandas as pd
import re
from datetime import datetime

def process_files(src_folder, dest_folder):
    # Шаблон имени файла: ГГГГ-ММ-ДД-ЧЧ-ММ-idмагазина.csv
    pattern = r'^\d{4}-\d{2}-\d{2}-\d{2}-\d{2}-\d+\.csv$'

    # Проверяем существование исходной папки
    if not os.path.exists(src_folder):
        print(f"Ошибка: Исходная папка '{src_folder}' не существует.")
        return

    # Создаём целевую папку, если её нет
    os.makedirs(dest_folder, exist_ok=True)

    # Список обязательных колонок
    required_columns = ['date', 'product', 'store', 'cost']

    # Получаем список всех файлов в исходной папке
    all_files = os.listdir(src_folder)
    valid_files = []

    for file in all_files:
        if re.match(pattern, file):
            valid_files.append(os.path.join(src_folder, file))

    if not valid_files:
        # Если подходящих файлов нет, создаём пустой CSV с обязательными колонками и расширением .csv
        output_path = os.path.join(dest_folder, 'combined_data.csv')
        pd.DataFrame(columns=required_columns).to_csv(
            output_path, index=False, sep=',', encoding='utf-8'
        )
        print(f"Предупреждение: В папке '{src_folder}' не найдено файлов, соответствующих шаблону. Создан пустой файл '{output_path}'.")
        return

    dataframes = []

    for file_path in valid_files:
        try:
            # Читаем файл
            df = pd.read_csv(file_path)

            # Извлекаем дату и время из имени файла для сортировки
            filename_without_ext = os.path.splitext(os.path.basename(file_path))[0]
            date_parts = filename_without_ext.split('-')
            date_str = f"{date_parts[0]}-{date_parts[1]}-{date_parts[2]}"  # ГГГГ-ММ-ДД
            time_str = f"{date_parts[3]}:{date_parts[4]}"  # ЧЧ:ММ
            datetime_str = f"{date_str} {time_str}"

            # Обрабатываем некорректные даты
            try:
                file_datetime = datetime.strptime(datetime_str, '%Y-%m-%d %H:%M')
            except ValueError as e:
                print(f"Пропущен файл {file_path} из‑за некорректной даты в имени: {e}")
                continue

            # Оставляем только обязательные колонки
            available_columns = [col for col in required_columns if col in df.columns]
            if not available_columns:
                print(f"Пропущен файл {file_path}: отсутствуют все обязательные колонки.")
                continue

            df_filtered = df[available_columns].copy()

            # Добавляем недостающие обязательные колонки как NaN
            for col in required_columns:
                if col not in df_filtered.columns:
                    df_filtered[col] = pd.NA

            # Восстанавливаем порядок колонок
            df_filtered = df_filtered[required_columns]

            # Добавляем колонку для сортировки
            df_filtered['file_datetime'] = file_datetime
            dataframes.append(df_filtered)

        except Exception as e:
            print(f"Ошибка при чтении файла {file_path}: {e}")
            continue

    if dataframes:
        # Объединяем все данные
        combined_df = pd.concat(dataframes, ignore_index=True)
        # Сортируем по дате из имени файла, затем по продукту
        sort_columns = ['file_datetime']
        if 'product' in combined_df.columns:
            sort_columns.append('product')
        combined_df.sort_values(by=sort_columns, inplace=True)
        # Удаляем служебную колонку
        combined_df.drop(columns=['file_datetime'], inplace=True, errors='ignore')
        # Сбрасываем индекс для корректного сравнения в тестах
        combined_df.reset_index(drop=True, inplace=True)
    else:
        combined_df = pd.DataFrame(columns=required_columns)

    # Создаём путь для выходного файла с расширением .csv
    output_path = os.path.join(dest_folder, 'combined_data.csv')
    # Сохраняем объединённые данные с разделителем запятая и расширением .csv
    combined_df.to_csv(output_path, index=False, sep=',', encoding='utf-8')
    print(f"Успешно обработано {len(valid_files)} файлов. Результат сохранён в '{output_path}'.")

# Пример вызова функции:
src_folder = 'reports-main'
dest_folder = 'comb_reports'
process_files(src_folder, dest_folder)


Пропущен файл reports-main/2023-02-17-05-38-2.csv: отсутствуют все обязательные колонки.
Пропущен файл reports-main/2023-02-15-10-26-0.csv: отсутствуют все обязательные колонки.
Пропущен файл reports-main/2023-02-16-02-55-1.csv: отсутствуют все обязательные колонки.
Пропущен файл reports-main/2023-02-19-11-05-4.csv: отсутствуют все обязательные колонки.
Пропущен файл reports-main/2023-02-30-02-55-23.csv из‑за некорректной даты в имени: day is out of range for month
Пропущен файл reports-main/2023-02-18-19-47-3.csv: отсутствуют все обязательные колонки.
Успешно обработано 6 файлов. Результат сохранён в 'comb_reports/combined_data.csv'.


✏️ ✏️ ✏️

**Проверка**

Чтобы проверить свое решение, запустите код в следующих ячейках

In [65]:
# Здесь будет скачиваться файл с эталонным ответом

!wget https://gist.github.com/Vs8th/9347dd7b8f59de2997feb19770dc32c1/raw/data.csv

import pandas as pd

user_answer = pd.read_csv(f'{dest_folder}/combined_data.csv')
correct_answer = pd.read_csv('data.csv')

--2026-03-05 13:09:06--  https://gist.github.com/Vs8th/9347dd7b8f59de2997feb19770dc32c1/raw/data.csv
Resolving gist.github.com (gist.github.com)... 20.27.177.113
Connecting to gist.github.com (gist.github.com)|20.27.177.113|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://gist.githubusercontent.com/Vs8th/9347dd7b8f59de2997feb19770dc32c1/raw/data.csv [following]
--2026-03-05 13:09:07--  https://gist.githubusercontent.com/Vs8th/9347dd7b8f59de2997feb19770dc32c1/raw/data.csv
Resolving gist.githubusercontent.com (gist.githubusercontent.com)... 185.199.111.133, 185.199.110.133, 185.199.108.133, ...
Connecting to gist.githubusercontent.com (gist.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 984 [text/plain]
Saving to: ‘data.csv.2’

data.csv.2          100%[===================>]     984  --.-KB/s    in 0s      

2026-03-05 13:09:07 (62.5 MB/s) - ‘data.csv.2’ saved [984/984]



In [66]:
try:
  assert (user_answer == correct_answer).all().all(), 'Ответы не совпадают'
  assert user_answer.columns.equals(correct_answer.columns), 'Названия столбцов не совпадают'
except Exception as err:
  raise AssertionError(f'При проверке возникла ошибка {repr(err)}')
else:
  print('Поздравляем, Вы справились и успешно прошли все проверки!')

AssertionError: При проверке возникла ошибка ValueError('Can only compare identically-labeled (both index and columns) DataFrame objects')